In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pydicom

In [ ]:
import os
from pathlib import Path
import pydicom
import numpy as np
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import pandas as pd
import albumentations as A
import cv2
import random
import csv
import torch
import torchvision.transforms as transforms

In [ ]:
# Alexandra file path
BASE_DIR = Path("/content/drive/MyDrive/Fall 2025/ML/Final/preprocessed-data")
file_path_train = '/content/drive/MyDrive/Fall 2025/ML/Final/rsna-pneumonia-detection-challenge/stage_2_train_images'
file_path_test = '/content/drive/MyDrive/Fall 2025/ML/Final/rsna-pneumonia-detection-challenge/stage_2_test_images'
train_labels_path = '/content/drive/MyDrive/Fall 2025/ML/Final/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv'

In [ ]:
# Amanda file path
BASE_DIR = Path("/content/drive/MyDrive/Final/preprocessed-data")
file_path_train = '/content/drive/MyDrive/Final/rsna-pneumonia-detection-challenge/stage_2_train_images'
file_path_test = '/content/drive/MyDrive/Final/rsna-pneumonia-detection-challenge/stage_2_test_images'
train_labels_path = '/content/drive/MyDrive/Final/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv'

In [ ]:
file_path_train = '/content/drive/My Drive/ORIE 5750 - Applied Machine Learning/Final/rsna-pneumonia-detection-challenge/stage_2_train_images'
file_path_test = '/content/drive/My Drive/ORIE 5750 - Applied Machine Learning/Final/rsna-pneumonia-detection-challenge/stage_2_test_images'
train_labels_path = "/content/drive/My Drive/ORIE 5750 - Applied Machine Learning/Final/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv"

In [ ]:
save_intermediate_dir = BASE_DIR / "images_512"
save_final_dir = BASE_DIR / "images_224"
bboxes_csv_path = BASE_DIR / "resized_bboxes_512.csv"
classification_csv_path = BASE_DIR / "classification_labels_224.csv"

save_intermediate_dir.mkdir(parents=True, exist_ok=True)
save_final_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
def load_rsna_labels(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df["Target"] == 1]

    labels = {}
    for _, row in df.iterrows():
        pid = row["patientId"]
        x_min = row["x"]
        y_min = row["y"]
        x_max = x_min + row["width"]
        y_max = y_min + row["height"]

        if pid not in labels:
            labels[pid] = []

        labels[pid].append([x_min, y_min, x_max, y_max])

    return labels

In [ ]:
with open(bboxes_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["patientId", "x_min", "y_min", "x_max", "y_max"])

In [ ]:
def load_dicom_as_pil(dcm_path):
    dicom = pydicom.dcmread(dcm_path)
    arr = dicom.pixel_array.astype(np.float32)

    arr -= arr.min()
    arr /= arr.max()
    arr *= 255.0

    return Image.fromarray(arr.astype(np.uint8)), arr.shape[0], arr.shape[1]

In [ ]:
def resize_bboxes(bboxes, old_w, old_h, new_w, new_h):
    scale_w = new_w / old_w
    scale_h = new_h / old_h

    new_boxes = []
    for box in bboxes:
        x_min, y_min, x_max, y_max = box
        x_min_new = x_min * scale_w
        y_min_new = y_min * scale_h
        x_max_new = x_max * scale_w
        y_max_new = y_max * scale_h
        new_boxes.append([x_min_new, y_min_new, x_max_new, y_max_new])

    return new_boxes

In [ ]:
augment_tfms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=5, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
],
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
)

NUM_AUG = 1

In [ ]:
preprocess_224 = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
def load_rsna_dataset(dcm_dir, labels_dict, resize_to=512, final_size=224):
    patient_ids = sorted(os.listdir(dcm_dir))

    all_tensors = []
    all_boxes = []
    all_ids = []
    classification_labels = []

    for idx, filename in enumerate(tqdm(patient_ids, desc="Processing RSNA images")):
        if not filename.endswith(".dcm"):
            continue

        patient_id = filename.replace(".dcm", "")
        dcm_path = os.path.join(dcm_dir, filename)

        pil_img, old_h, old_w = load_dicom_as_pil(dcm_path)

        pil_img_resized = pil_img.resize((resize_to, resize_to))

        pil_img_resized.save(save_intermediate_dir / f"{patient_id}.png")

        orig_boxes = labels_dict.get(patient_id, [])
        resized_boxes = resize_bboxes(orig_boxes, old_w, old_h, resize_to, resize_to)

        with open(bboxes_csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            for box in resized_boxes:
                writer.writerow([patient_id, *box])

        tensor_img = preprocess_224(pil_img_resized)

        img_uint8 = (tensor_img.permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
        Image.fromarray(img_uint8).save(save_final_dir / f"{patient_id}.png")

        all_tensors.append(tensor_img)
        all_boxes.append(resized_boxes)
        all_ids.append(patient_id)

        label = 1 if len(resized_boxes) > 0 else 0
        classification_labels.append((patient_id, label))

        if idx % 5 == 0 and NUM_AUG > 0:
          img_np = np.array(pil_img_resized)
          for i in range(NUM_AUG):
              if len(resized_boxes) > 0:
                  labels_for_aug = [1]*len(resized_boxes)
                  bboxes_for_aug = resized_boxes
              else:
                  labels_for_aug = []
                  bboxes_for_aug = []

              augmented = augment_tfms(image=img_np, bboxes=bboxes_for_aug, labels=labels_for_aug)
              aug_img = augmented["image"]
              aug_boxes = augmented["bboxes"]

              pil_aug = Image.fromarray(aug_img)
              aug_id = f"{patient_id}_aug{i+1}"

              pil_aug.save(save_intermediate_dir / f"{aug_id}.png")

              with open(bboxes_csv_path, "a", newline="") as f:
                  writer = csv.writer(f)
                  for box in aug_boxes:
                      writer.writerow([aug_id, *box])

              tensor_aug = preprocess_224(pil_aug)
              img_aug_uint8 = (tensor_aug.permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
              Image.fromarray(img_aug_uint8).save(save_final_dir / f"{aug_id}.png")

              classification_labels.append((aug_id, label))

              all_tensors.append(tensor_aug)
              all_boxes.append(aug_boxes)
              all_ids.append(aug_id)

    with open(classification_csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["patientId", "label"])
        writer.writerows(classification_labels)

    batch_tensor = torch.stack(all_tensors)

    return batch_tensor, all_boxes, all_ids


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Processing RSNA images:  58%|█████▊    | 15458/26699 [50:55<10:26:28,  3.34s/it]

In [ ]:
labels_dict = load_rsna_labels(train_labels_path)

images_224, boxes_224, ids = load_rsna_dataset(
    dcm_dir=file_path_train,
    labels_dict=labels_dict,
    resize_to=512,
    final_size=224
)

print("Final image batch:", images_224.shape)
print("Example bboxes for first image:", boxes_224[0])
print("Patient ID:", ids[0])

##JUST Preprocessing (for test dataset)

In [ ]:
dcm_dir = Path("path/to/rsna/dicom")
train_labels_path = Path("path/to/rsna_labels.csv")
save_intermediate_dir = Path("intermediate_512")
save_final_dir = Path("final_224")
bboxes_csv_path = Path("resized_bboxes.csv")
classification_csv_path = Path("classification_labels.csv")

save_intermediate_dir.mkdir(parents=True, exist_ok=True)
save_final_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load Bounding Box CSV

def load_rsna_labels(csv_path):
    df = pd.read_csv(csv_path)
    df = df[df["Target"] == 1]

    labels = {}
    for _, row in df.iterrows():
        pid = row["patientId"]
        x_min = row["x"]
        y_min = row["y"]
        x_max = x_min + row["width"]
        y_max = y_min + row["height"]

        if pid not in labels:
            labels[pid] = []

        labels[pid].append([x_min, y_min, x_max, y_max])

    return labels

with open(bboxes_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["patientId", "x_min", "y_min", "x_max", "y_max"])

In [ ]:
# DICOM to PIL image

def load_dicom_as_pil(dcm_path):
    dicom = pydicom.dcmread(dcm_path)
    arr = dicom.pixel_array.astype(np.float32)

    # normalize 0–255
    arr -= arr.min()
    arr /= arr.max()
    arr *= 255.0

    return Image.fromarray(arr.astype(np.uint8)), arr.shape[0], arr.shape[1]

In [ ]:
# Resize bounding boxes

def resize_bboxes(bboxes, old_w, old_h, new_w, new_h):
    """
    Rescales bounding boxes to new size.
    Each bbox is [x_min, y_min, x_max, y_max]
    """
    scale_w = new_w / old_w
    scale_h = new_h / old_h

    new_boxes = []
    for box in bboxes:
        x_min, y_min, x_max, y_max = box
        new_boxes.append([
            x_min * scale_w,
            y_min * scale_h,
            x_max * scale_w,
            y_max * scale_h
        ])

    return new_boxes

In [ ]:
# Preprocess for ResNet50 / DenseNet121

preprocess_224 = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [ ]:
# Load RSNA DICOMs

def load_rsna_dataset(dcm_dir, labels_dict, resize_to=512, final_size=224):
    patient_files = sorted([f for f in os.listdir(dcm_dir) if f.endswith(".dcm")])

    all_tensors = []
    all_boxes = []
    all_ids = []
    classification_labels = []

    for filename in tqdm(patient_files, desc="Processing RSNA images"):

        patient_id = filename.replace(".dcm", "")
        dcm_path = os.path.join(dcm_dir, filename)

        pil_img, old_h, old_w = load_dicom_as_pil(dcm_path)

        pil_img_resized = pil_img.resize((resize_to, resize_to))

        pil_img_resized.save(save_intermediate_dir / f"{patient_id}.png")

        orig_boxes = labels_dict.get(patient_id, [])
        resized_boxes = resize_bboxes(orig_boxes, old_w, old_h, resize_to, resize_to)

        with open(bboxes_csv_path, "a", newline="") as f:
            writer = csv.writer(f)
            for box in resized_boxes:
                writer.writerow([patient_id, *box])

        tensor_img = preprocess_224(pil_img_resized)

        img_uint8 = (tensor_img.permute(1, 2, 0).numpy() * 255.0).clip(0, 255).astype(np.uint8)
        Image.fromarray(img_uint8).save(save_final_dir / f"{patient_id}.png")

        all_tensors.append(tensor_img)
        all_boxes.append(resized_boxes)
        all_ids.append(patient_id)

        label = 1 if len(resized_boxes) > 0 else 0
        classification_labels.append((patient_id, label))

    with open(classification_csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["patientId", "label"])
        writer.writerows(classification_labels)

    images_tensor = torch.stack(all_tensors)

    return images_tensor, all_boxes, all_ids

In [ ]:
# Ex

labels_dict = load_rsna_labels(train_labels_path)

images_224, boxes_224, ids = load_rsna_dataset(
    dcm_dir=dcm_dir,
    labels_dict=labels_dict,
    resize_to=512,
    final_size=224
)

print("Final image batch:", images_224.shape)
print("Example bboxes for first image:", boxes_224[0])
print("Patient ID:", ids[0])